# 2차원 파동을 학습하는 PINN

이 실습에서는 파동 방정식, 초기조건, 경계조건을 손실함수로 표현합니다. 준비된 학습 코드를 실행한 뒤 파동 속도를 바꾸고, 예측과 해석해의 차이를 비교합니다.

**준비:** [환경 확인](00_환경확인.ipynb)을 마친 행사 GPU 환경. 이 초안은 PhysicsNeMo 25.11의 Sym API를 사용합니다. 학습 시간과 수렴 정도는 행사 GPU 리허설에서 확인해야 합니다.

**완료 기준:** 예측 그림과 `metrics.json`을 만들고, 파동 속도를 바꾼 두 실험의 RMSE와 PDE 잔차를 구분해 설명합니다. 학습을 끝냈다는 사실만으로 정확한 해를 얻었다고 판단하지 않습니다.

[수업 안내로 돌아가기](README.md)


## 문제와 정답

공간은 $0\le x,y\le\pi$, 시간은 $0\le t\le2$입니다. 네 경계의 변위는 0이며, 내부에서는 다음 식을 만족합니다.

$$u_{tt}=c^2(u_{xx}+u_{yy}),\qquad c>0$$

$$u(x,y,0)=\sin x\sin y,\qquad u_t(x,y,0)=\sin x\sin y$$

이 조건에 맞는 해석해는 다음과 같습니다.

$$u(x,y,t)=\sin x\sin y\left[\cos(\omega t)+\frac{\sin(\omega t)}{\omega}\right],\qquad \omega=\sqrt2c$$

`wave_reference.py`가 이 해석해를 계산합니다. 모델은 학습할 때 PDE와 초기·경계조건을 사용하며, 해석해는 검증에 사용합니다.


In [ ]:
from pathlib import Path
import json
import sys
import numpy as np
import matplotlib.pyplot as plt

search_roots = [Path.cwd(), *Path.cwd().parents]
WAVE_DIR = next((p / "ai4sci" / "wave" for p in search_roots
                 if (p / "ai4sci" / "wave" / "wave_baseline.py").is_file()), None)
if WAVE_DIR is None:
    raise RuntimeError("저장소의 ai4sci 폴더에서 이 노트북을 열어 주세요.")
sys.path.insert(0, str(WAVE_DIR))
from wave_reference import exact_solution
print("실습 코드:", WAVE_DIR.name)


## 학습 전에 파동 확인

아래 그림은 학습 모델이 아닌 해석해입니다. 같은 공간 패턴이 시간에 따라 진동하며, 속도 `c`가 클수록 진동이 빨라집니다. 그림의 색은 변위 $u$를 나타냅니다.


In [ ]:
axis = np.linspace(0, np.pi, 60)
X, Y = np.meshgrid(axis, axis, indexing="ij")
fig, axes = plt.subplots(1, 3, figsize=(12, 3), constrained_layout=True)
for ax, moment in zip(axes, [0.0, 1.0, 2.0]):
    im = ax.imshow(exact_solution(X, Y, moment, c=1.0), origin="lower",
                   extent=[0, np.pi, 0, np.pi], vmin=-1.3, vmax=1.3, cmap="RdBu_r")
    ax.set(title=f"Exact solution, t={moment}", xlabel="y", ylabel="x")
fig.colorbar(im, ax=axes, label="Displacement u")
plt.show()


## 조건과 코드의 연결

[`wave_baseline.py`](wave/wave_baseline.py)에서 다음 부분을 찾아봅니다.

| 물리적 의미 | 코드 | 손실의 목표 |
|---|---|---|
| 내부의 파동 방정식 | `WaveEquation2D`와 `interior` | `wave_equation`을 0으로 |
| 시작 시점의 변위와 속도 | `initial` | `u`, `u__t`를 초기조건으로 |
| 네 변의 고정 경계 | `boundary` | 경계의 `u`를 0으로 |
| 정답과의 비교 | `reference` | 예측과 해석해의 오차 측정 |

파동 속도는 아래 `SPEED`, 학습 반복 횟수는 `STEPS`에서 바꿉니다. 실험마다 `RUN_NAME`을 바꾸면 결과를 각각 보관할 수 있습니다.


In [ ]:
RUN_NAME = "baseline_01"
SPEED = 1.0
STEPS = 1000


## 학습 실행

`%cd`는 이 노트북의 작업 폴더를 실습 코드 위치로 바꿉니다. `!python`은 Python 학습 파일을 실행하는 터미널 명령입니다. 학습 로그가 출력된 뒤 마지막에 `Results:`와 결과 폴더가 표시되면 다음 셀로 이동합니다.

오류가 나면 마지막 오류 문장을 강사에게 보여 주세요. 같은 실험명을 다시 쓰면 이전 결과를 보존하기 위해 중단됩니다. 새 `RUN_NAME`을 지정해 다시 실행할 수 있습니다. 제한 시간 안에 끝나지 않으면 강사가 준비한 결과를 함께 해석합니다.


In [ ]:
RUN_COMPLETED = False
result_dir = WAVE_DIR / "runs" / RUN_NAME
if result_dir.exists():
    raise RuntimeError("이미 사용한 실험명입니다. RUN_NAME을 바꾼 뒤 이 셀을 다시 실행해 주세요.")
%cd {WAVE_DIR}
!{sys.executable} wave_baseline.py custom.run_name={RUN_NAME} custom.c={SPEED} training.max_steps={STEPS}
if get_ipython().user_ns.get("_exit_code") != 0:
    raise RuntimeError("학습 명령이 실패했습니다. 위 오류를 확인한 뒤 새 실험명으로 실행해 주세요.")
RUN_COMPLETED = True


## 예측과 해석해 비교

왼쪽은 해석해, 가운데는 PINN 예측, 오른쪽은 절대오차입니다. 같은 색 범위로 해석해와 예측을 비교합니다. RMSE는 검증 격자 전체의 오차이고, PDE 잔차는 별도로 뽑은 내부 128점에서 방정식을 얼마나 잘 만족하는지 나타냅니다. 두 지표를 함께 읽습니다.


In [ ]:
if not globals().get("RUN_COMPLETED", False):
    raise RuntimeError("현재 학습 실행이 완료되지 않았습니다. 이전 결과를 새 실험으로 읽지 않도록 중단합니다.")
result_dir = WAVE_DIR / "runs" / RUN_NAME
if not (result_dir / "metrics.json").is_file():
    raise RuntimeError("완료된 결과가 없습니다. 위 학습 셀의 오류 또는 진행 상태를 확인해 주세요.")
metrics = json.loads((result_dir / "metrics.json").read_text())
if metrics["c"] != SPEED or metrics["training_steps_requested"] != STEPS:
    raise RuntimeError("결과의 속도·반복 횟수가 현재 설정과 다릅니다. 학습 셀을 다시 확인해 주세요.")
print(json.dumps(metrics, indent=2))
data = np.load(result_dir / "prediction.npz")
time_values = np.unique(data["t"])
slice_time = time_values[len(time_values)//2]
mask = np.isclose(data["t"].ravel(), slice_time)
truth = data["truth"].ravel()[mask].reshape(20, 20)
prediction = data["prediction"].ravel()[mask].reshape(20, 20)
limit = max(abs(truth).max(), abs(prediction).max(), 1e-6)
fig, axes = plt.subplots(1, 3, figsize=(12, 3), constrained_layout=True)
for ax, title, values in zip(axes[:2], ["Exact solution", "PINN prediction"], [truth, prediction]):
    im = ax.imshow(values, origin="lower", extent=[0,np.pi,0,np.pi],
                   vmin=-limit, vmax=limit, cmap="RdBu_r")
    ax.set(title=f"{title}, t={slice_time:.2f}", xlabel="y", ylabel="x")
fig.colorbar(im, ax=axes[:2], label="Displacement u")
error = axes[2].imshow(abs(truth-prediction), origin="lower", extent=[0,np.pi,0,np.pi], cmap="magma")
axes[2].set(title="Absolute error", xlabel="y", ylabel="x")
fig.colorbar(error, ax=axes[2], label="Absolute error")
plt.show()


## 두 번째 실험

`RUN_NAME`을 `speed_05`, `SPEED`를 `0.5`로 바꾸고 학습 셀부터 다시 실행합니다. 같은 반복 횟수에서 결과를 비교한 뒤, 시간이 남으면 새 실험명으로 `STEPS`를 늘려봅니다.

- 파동 속도가 바뀌면 시간 방향 진동은 어떻게 달라지나요?
- RMSE가 작아질 때 PDE 잔차도 함께 작아지나요?
- 반복 횟수를 늘리면 오차가 줄어드는지 실제 결과로 확인했나요?

실험 결과는 `wave/runs/<실험명>/prediction.npz`와 `metrics.json`에 남습니다. 수업에서는 서로 다른 속도·반복 횟수의 점수를 단일 순위로 비교하지 않습니다.

[수업 안내로 돌아가기](README.md)
